# English to Arabic Translation using Local Ollama 
Reads the parquet file, translates the `query` column, and saves results to a CSV file.

In [ ]:
import pandas as pd
import requests
import sys
import json
import time
import os
from pathlib import Path

# ─────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────
OLLAMA_URL     = "http://localhost:11434/api/generate"         # Local Ollama endpoint

SYSTEM_PROMPT  = (
    "You are a professional English-to-Arabic translator. "
    "Translate the following text into Modern Standard Arabic, "
    "maintaining the formal tone and ensuring grammatical correctness. "
    "Output ONLY the Arabic translation of the query . If it is a question only translate it and do not give the answer. If the answer is a number give it to me in english numerals do not spell it out. "
    "Do not include any explanation, commentary, transliteration, or any other text besides the translation itself."
)

# ─────────────────────────────────────────────────
# TRANSLATION FUNCTION
# ─────────────────────────────────────────────────
def translate_to_arabic(text: str) -> tuple[str, float]:
    """
    Send a query to local Ollama and return (translation, elapsed_seconds).
    """
    payload = {
        "model": OLLAMA_MODEL,
        "system": SYSTEM_PROMPT,
        "prompt": str(text),
        "stream": False,
        "options": {
            "temperature": 0.1 ,  # Low temperature for deterministic translation
        }
    }

    start = time.perf_counter()
    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=120)
        response.raise_for_status()
        result = response.json()
        translation = result.get("response", "").strip()
    except requests.exceptions.RequestException as e:
        translation = f"ERROR: {e}"
    elapsed = round(time.perf_counter() - start, 3)

    return translation, elapsed

# ─────────────────────────────────────────────────
# OLLAMA PINGER FUNCTION
# ─────────────────────────────────────────────────
def check_ollama_ready(model_name):
    """Pings the local Ollama server to ensure it is running and the model exists."""
    print("Checking Ollama server status...")
    try:
        # Ping the local Ollama API
        response = requests.get('http://localhost:11434/api/tags', timeout=3)
        response.raise_for_status()
        
        # Verify the specific model is downloaded
        downloaded_models = [m['name'] for m in response.json().get('models', [])]
        
        # Ollama often appends tags like ':latest', so we check for partial matches
        if not any(model_name in m for m in downloaded_models):
            print(f"❌ Error: Ollama is running, but the model '{model_name}' was not found.")
            print(f"   Please run `ollama pull {model_name}` in your terminal.")
            sys.exit(1)
            
        print(f"✅ Ollama server is active and '{model_name}' is ready.\n")
        
    except requests.exceptions.ConnectionError:
        print("❌ Error: Could not connect to Ollama.")
        print("   Please make sure the Ollama application is running on your system.")
        sys.exit(1)
    except Exception as e:
        print(f"❌ Unexpected error connecting to Ollama: {e}")
        sys.exit(1)

# ─────────────────────────────────────────────────
# NUMBER TRANSLATOR FUNCTION
# ─────────────────────────────────────────────────

# Mapping from Western Arabic digits to Eastern Arabic-Indic digits
DIGIT_MAP = str.maketrans("0123456789", "٠١٢٣٤٥٦٧٨٩")


def to_arabic_numerals(value: str) -> str:
    """Translate digit characters in *value* to Eastern Arabic-Indic digits."""
    return value.translate(DIGIT_MAP)

## ChartQA

In [ ]:
# ─────────────────────────────────────────────────
# LOAD PARQUET
# ─────────────────────────────────────────────────
PARQUET_FILE   = r"D:\Users\Desktop\Thesis\drive datsets\chartqa.parquet"  # Path to parquet file
df = pd.read_parquet(PARQUET_FILE)
print(f"Loaded {len(df):,} rows")
print(f"Columns: {df.columns.tolist()}")

### aya

In [ ]:
# ─────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────
OLLAMA_MODEL   = "aya-expanse"

# TRIGGER FAILSAFE HERE: Stop execution immediately if Ollama is down
check_ollama_ready(OLLAMA_MODEL)

OUTPUT_CSV     = rf"d:\Users\Desktop\Thesis\{OLLAMA_MODEL}translation.csv" 

# Import numeric-check helpers from ground.py
import importlib.util
_spec = importlib.util.spec_from_file_location('ground', r'd:\Users\Desktop\Thesis\ground.py')
_mod  = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_mod)
is_numeric         = _mod.is_numeric
to_arabic_numerals = _mod.to_arabic_numerals

# ─────────────────────────────────────────────────
# RESUME SUPPORT — skip already translated rows
# ─────────────────────────────────────────────────
columns = [
    "id", "imgname", "query", "arabic_query", 
    "ground_truth", "arabic_ground_truth", 
    "type", "translation_time_sec"
]

if Path(OUTPUT_CSV).exists():
    existing = pd.read_csv(OUTPUT_CSV)
    done_ids = set(existing["id"].tolist())
    print(f"Resuming: {len(done_ids):,} rows already translated, skipping them.")
else:
    existing = None
    done_ids = set()
    pd.DataFrame(columns=columns).to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print("Starting fresh — output CSV created.")
    
# ─────────────────────────────────────────────────
# MAIN LOOP
# ─────────────────────────────────────────────────
SAMPLE_SIZE = None   # ← Set to an integer to process only the first N rows, or None to process ALL rows

rows_to_process = df[~df.index.isin(done_ids)].copy()
if SAMPLE_SIZE is not None:
    rows_to_process = rows_to_process.head(SAMPLE_SIZE)
    print(f"Sample mode: processing first {SAMPLE_SIZE} rows.")

total = len(rows_to_process)
print(f"Translating {total:,} rows...\n")

BATCH_SIZE = 5   # Append to CSV every N rows to avoid data loss on interruption
batch = []

# Assuming 'translate_to_arabic' is defined elsewhere in your environment
for i, (idx, row) in enumerate(rows_to_process.iterrows(), start=1):
    # Extract the 5 original columns
    imgname = row["imgname"]
    query = str(row["query"])
    ground_truth = str(row["label"]) # renaming 'label' to 'ground_truth'
    type_val = row["type"]
    
    # 1. Translate the query
    arabic_query, elapsed_q = translate_to_arabic(query)
    
    # 2. Translate the ground truth — skip model if it is a number
    if is_numeric(ground_truth):
        arabic_gt, elapsed_gt = to_arabic_numerals(ground_truth), 0
    else:
        arabic_gt, elapsed_gt = translate_to_arabic(ground_truth)

    # Combine the translation times for the row record
    total_elapsed = elapsed_q + elapsed_gt

    batch.append({
        "id": idx,
        "imgname": imgname,
        "query": query,
        "ground_truth": ground_truth,
        "type": type_val,
        "arabic_query": arabic_query,
        "arabic_ground_truth": arabic_gt,
        "translation_time_sec": round(total_elapsed, 2)
    })

    # Progress print (shortened text for clean console output)
    print(f"[{i}/{total}] id={idx} | {total_elapsed:.2f}s | Q: {query[:30]}... | GT: {ground_truth[:30]}...")

    # Flush batch to disk
    if i % BATCH_SIZE == 0 or i == total:
        # Enforcing column order to prevent misalignment (as discussed previously)
        batch_df = pd.DataFrame(batch)[columns] 
        batch_df.to_csv(OUTPUT_CSV, mode="a", header=False, index=False, encoding="utf-8-sig")
        batch.clear()
        print(f"  → Saved {i:,}/{total:,} rows to {OUTPUT_CSV}")

print("\n✅ Translation complete!")

### qwen3

In [ ]:
# ─────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────
OLLAMA_MODEL   = "qwen3:8b"

# TRIGGER FAILSAFE HERE: Stop execution immediately if Ollama is down
check_ollama_ready(OLLAMA_MODEL)

OUTPUT_CSV     = rf"d:\Users\Desktop\Thesis\qwen3translation.csv" 

# Import numeric-check helpers from ground.py
import importlib.util
_spec = importlib.util.spec_from_file_location('ground', r'd:\Users\Desktop\Thesis\ground.py')
_mod  = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_mod)
is_numeric         = _mod.is_numeric
to_arabic_numerals = _mod.to_arabic_numerals

# ─────────────────────────────────────────────────
# RESUME SUPPORT — skip already translated rows
# ─────────────────────────────────────────────────
columns = [
    "id", "imgname", "query", "arabic_query", 
    "ground_truth", "arabic_ground_truth", 
    "type", "translation_time_sec"
]

if Path(OUTPUT_CSV).exists():
    existing = pd.read_csv(OUTPUT_CSV)
    done_ids = set(existing["id"].tolist())
    print(f"Resuming: {len(done_ids):,} rows already translated, skipping them.")
else:
    existing = None
    done_ids = set()
    pd.DataFrame(columns=columns).to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print("Starting fresh — output CSV created.")
    
# ─────────────────────────────────────────────────
# MAIN LOOP
# ─────────────────────────────────────────────────
SAMPLE_SIZE = None   # ← Set to an integer to process only the first N rows, or None to process ALL rows

rows_to_process = df[~df.index.isin(done_ids)].copy()
if SAMPLE_SIZE is not None:
    rows_to_process = rows_to_process.head(SAMPLE_SIZE)
    print(f"Sample mode: processing first {SAMPLE_SIZE} rows.")

total = len(rows_to_process)
print(f"Translating {total:,} rows...\n")

BATCH_SIZE = 5   # Append to CSV every N rows to avoid data loss on interruption
batch = []

# Assuming 'translate_to_arabic' is defined elsewhere in your environment
for i, (idx, row) in enumerate(rows_to_process.iterrows(), start=1):
    # Extract the 5 original columns
    imgname = row["imgname"]
    query = str(row["query"])
    ground_truth = str(row["label"]) # renaming 'label' to 'ground_truth'
    type_val = row["type"]
    
    # 1. Translate the query
    arabic_query, elapsed_q = translate_to_arabic(query)
    
    # 2. Translate the ground truth — skip model if it is a number
    if is_numeric(ground_truth):
        arabic_gt, elapsed_gt = to_arabic_numerals(ground_truth), 0
    else:
        arabic_gt, elapsed_gt = translate_to_arabic(ground_truth)

    # Combine the translation times for the row record
    total_elapsed = elapsed_q + elapsed_gt

    batch.append({
        "id": idx,
        "imgname": imgname,
        "query": query,
        "ground_truth": ground_truth,
        "type": type_val,
        "arabic_query": arabic_query,
        "arabic_ground_truth": arabic_gt,
        "translation_time_sec": round(total_elapsed, 2)
    })

    # Progress print (shortened text for clean console output)
    print(f"[{i}/{total}] id={idx} | {total_elapsed:.2f}s | Q: {query[:30]}... | GT: {ground_truth[:30]}...")

    # Flush batch to disk
    if i % BATCH_SIZE == 0 or i == total:
        # Enforcing column order to prevent misalignment (as discussed previously)
        batch_df = pd.DataFrame(batch)[columns] 
        batch_df.to_csv(OUTPUT_CSV, mode="a", header=False, index=False, encoding="utf-8-sig")
        batch.clear()
        print(f"  → Saved {i:,}/{total:,} rows to {OUTPUT_CSV}")

print("\n✅ Translation complete!")

### Gemma 4

In [ ]:
# ─────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────
OLLAMA_MODEL   = "gemma4:e4b"

# TRIGGER FAILSAFE HERE: Stop execution immediately if Ollama is down
check_ollama_ready(OLLAMA_MODEL)

OUTPUT_CSV     = rf"d:\Users\Desktop\Thesis\gemma4translation.csv" 

# Import numeric-check helpers from ground.py
import importlib.util
_spec = importlib.util.spec_from_file_location('ground', r'd:\Users\Desktop\Thesis\ground.py')
_mod  = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_mod)
is_numeric         = _mod.is_numeric
to_arabic_numerals = _mod.to_arabic_numerals

# ─────────────────────────────────────────────────
# RESUME SUPPORT — skip already translated rows
# ─────────────────────────────────────────────────
columns = [
    "id", "imgname", "query", "arabic_query", 
    "ground_truth", "arabic_ground_truth", 
    "type", "translation_time_sec"
]

if Path(OUTPUT_CSV).exists():
    existing = pd.read_csv(OUTPUT_CSV)
    done_ids = set(existing["id"].tolist())
    print(f"Resuming: {len(done_ids):,} rows already translated, skipping them.")
else:
    existing = None
    done_ids = set()
    pd.DataFrame(columns=columns).to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print("Starting fresh — output CSV created.")
    
# ─────────────────────────────────────────────────
# MAIN LOOP
# ─────────────────────────────────────────────────
SAMPLE_SIZE = None   # ← Set to an integer to process only the first N rows, or None to process ALL rows

rows_to_process = df[~df.index.isin(done_ids)].copy()
if SAMPLE_SIZE is not None:
    rows_to_process = rows_to_process.head(SAMPLE_SIZE)
    print(f"Sample mode: processing first {SAMPLE_SIZE} rows.")

total = len(rows_to_process)
print(f"Translating {total:,} rows...\n")

BATCH_SIZE = 5   # Append to CSV every N rows to avoid data loss on interruption
batch = []

# Assuming 'translate_to_arabic' is defined elsewhere in your environment
for i, (idx, row) in enumerate(rows_to_process.iterrows(), start=1):
    # Extract the 5 original columns
    imgname = row["imgname"]
    query = str(row["query"])
    ground_truth = str(row["label"]) # renaming 'label' to 'ground_truth'
    type_val = row["type"]
    
    # 1. Translate the query
    arabic_query, elapsed_q = translate_to_arabic(query)
    
    # 2. Translate the ground truth — skip model if it is a number
    if is_numeric(ground_truth):
        arabic_gt, elapsed_gt = to_arabic_numerals(ground_truth), 0
    else:
        arabic_gt, elapsed_gt = translate_to_arabic(ground_truth)

    # Combine the translation times for the row record
    total_elapsed = elapsed_q + elapsed_gt

    batch.append({
        "id": idx,
        "imgname": imgname,
        "query": query,
        "ground_truth": ground_truth,
        "type": type_val,
        "arabic_query": arabic_query,
        "arabic_ground_truth": arabic_gt,
        "translation_time_sec": round(total_elapsed, 2)
    })

    # Progress print (shortened text for clean console output)
    print(f"[{i}/{total}] id={idx} | {total_elapsed:.2f}s | Q: {query[:30]}... | GT: {ground_truth[:30]}...")

    # Flush batch to disk
    if i % BATCH_SIZE == 0 or i == total:
        # Enforcing column order to prevent misalignment (as discussed previously)
        batch_df = pd.DataFrame(batch)[columns] 
        batch_df.to_csv(OUTPUT_CSV, mode="a", header=False, index=False, encoding="utf-8-sig")
        batch.clear()
        print(f"  → Saved {i:,}/{total:,} rows to {OUTPUT_CSV}")

print("\n✅ Translation complete!")

### Finale

In [ ]:
import pandas as pd

# Define your file path
# CSV_PATH = r"d:\Users\Desktop\Thesis\ChartTrans\aya-expansetranslation.csv"
# CSV_PATH = r"d:\Users\Desktop\Thesis\ChartTrans\gemma4translation.csv"
CSV_PATH = r"d:\Users\Desktop\Thesis\ChartTrans\qwen3translation.csv"

# Load the CSV file
df = pd.read_csv(CSV_PATH)

# Condition 1: The original English 'query' DOES end with a standard '?'
is_english_question = df["query"].fillna("").astype(str).str.strip().str.endswith("?")

# Condition 2: The translated 'arabic_query' DOES NOT end with an Arabic '؟'
missing_arabic_qm = ~df["arabic_query"].fillna("").astype(str).str.strip().str.endswith("؟")

# Combine both conditions using the '&' (AND) operator
# This filters for rows where BOTH conditions are true at the same time
problematic_rows = df[is_english_question & missing_arabic_qm]

# Get the total count and the specific IDs
missing_count = len(problematic_rows)
problematic_ids = problematic_rows["id"].tolist()

# Print the results
print(f"Total rows where English had '?' but Arabic is missing '؟': {missing_count}")

if missing_count > 0:
    print("\nList of IDs with missing question marks:")
    # Print the IDs comma-separated for easy reading
    print(", ".join(map(str, problematic_ids)))

In [ ]:
# Define your file path
# CSV_PATH = r"d:\Users\Desktop\Thesis\ChartTrans\aya-expansetranslation.csv"
# CSV_PATH = r"d:\Users\Desktop\Thesis\ChartTrans\gemma4translation.csv"
CSV_PATH = r"d:\Users\Desktop\Thesis\ChartTrans\qwen3translation.csv"

# Load the CSV file
df = pd.read_csv(CSV_PATH)

# Define the exact name of your time column
TIME_COL = "translation_time_sec" 

# 1. Isolate the time column and drop any completely blank/missing rows
times = df[TIME_COL].dropna()

# 2. Calculate the Interquartile Range (IQR)
Q1 = times.quantile(0.25) # 25th percentile
Q3 = times.quantile(0.75) # 75th percentile
IQR = Q3 - Q1

# 3. Define the mathematical bounds for what constitutes an "outlier"
# The 1.5 multiplier is the industry standard for identifying outliers
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# 4. Filter the data to ONLY keep values inside the bounds
filtered_times = times[(times >= lower_bound) & (times <= upper_bound)]

# 5. Calculate the true average without the outliers
true_average = filtered_times.mean()

# Print the results clearly
outliers_removed = len(times) - len(filtered_times)

print("--- Translation Time Analysis ---")
print(f"Original row count  : {len(times):,}")
print(f"Rows kept (clean)   : {len(filtered_times):,}")
print(f"Outliers removed    : {outliers_removed:,}")
print(f"---------------------------------")
print(f"Raw Average (w/ outliers)   : {times.mean():.2f} seconds")
print(f"Clean Average (no outliers) : {true_average:.2f} seconds")

In [ ]:
# ─────────────────────────────────────────────────
# FINALE — Merge translations into a new parquet file
# ─────────────────────────────────────────────────
import pandas as pd

PARQUET_IN  = r"d:\Users\Desktop\Thesis\chartqa.parquet"
CSV_IN      = r"d:\Users\Desktop\Thesis\gemma4translation.csv"
PARQUET_OUT = r"d:\Users\Desktop\Thesis\chartqa_arabic.parquet"

# Load both files
df_pq  = pd.read_parquet(PARQUET_IN)
df_csv = pd.read_csv(CSV_IN, dtype=str)

# Keep only the translation columns we need from the CSV
df_trans = df_csv[["imgname", "query", "arabic_query", "arabic_ground_truth"]].copy()

# 1. Clean strings using TEMPORARY columns to ensure perfect matches
# This protects your original data from being altered in any way.
df_pq["_merge_imgname"] = df_pq["imgname"].astype(str).str.strip()
df_pq["_merge_query"] = df_pq["query"].astype(str).str.strip()

df_trans["_merge_imgname"] = df_trans["imgname"].astype(str).str.strip()
df_trans["_merge_query"] = df_trans["query"].astype(str).str.strip()

# 2. Add a temporary counter to distinguish duplicates
df_pq["occurrence"] = df_pq.groupby(["_merge_imgname", "_merge_query"]).cumcount()
df_trans["occurrence"] = df_trans.groupby(["_merge_imgname", "_merge_query"]).cumcount()

# 3. Merge using the temporary keys
# We drop the CSV's original imgname/query so they don't overwrite the Parquet's keys
df_merged = df_pq.merge(
    df_trans.drop(columns=["imgname", "query"]), 
    on=["_merge_imgname", "_merge_query", "occurrence"], 
    how="left"
)

# 4. Clean up all temporary columns
df_merged = df_merged.drop(columns=["_merge_imgname", "_merge_query", "occurrence"])

print(f"Original Parquet rows : {len(df_pq):,}")
print(f"Merged Parquet rows   : {len(df_merged):,} (Should match original!)")

matched = df_merged["arabic_query"].notna().sum()
print(f"Matched translations  : {matched:,} / {len(df_merged):,}")

# Save as parquet
df_merged.to_parquet(PARQUET_OUT, index=False, compression="zstd")
print(f"\n✅ Saved to: {PARQUET_OUT}")

# ─────────────────────────────────────────────────
# VERIFICATION STEP — Ensure original data is untouched
# ─────────────────────────────────────────────────
print("\n🔍 Running strict data verification...")

# Load both files straight from the hard drive for an unbiased check
df_original_check = pd.read_parquet(PARQUET_IN)
df_new_check = pd.read_parquet(PARQUET_OUT)

# Get the list of original columns
original_cols = df_original_check.columns.tolist()

try:
    # assert_frame_equal strictly checks that values, datatypes, and order are 100% identical
    pd.testing.assert_frame_equal(
        df_original_check[original_cols], 
        df_new_check[original_cols],
        check_dtype=True,
        check_exact=True
    )
    print("✅ SUCCESS: All original columns are 100% identical and untouched!")
except AssertionError as e:
    print("❌ WARNING: The original data was altered during the process.")
    print(f"Details: {e}")

df = pd.read_parquet(r"D:\Users\Desktop\Thesis\chartqa_arabic.parquet")
print(f"Loaded {len(df):,} rows")
print(f"Columns: {df.columns.tolist()}")

## MathVision

In [ ]:
# ─────────────────────────────────────────────────
# LOAD MATHVISION PARQUET
# ─────────────────────────────────────────────────
MATHVISION_PARQUET = r"D:\Users\Desktop\Thesis\drive datsets\mathvision.parquet"

mv_df = pd.read_parquet(MATHVISION_PARQUET)
print(f"Loaded {len(mv_df):,} rows")
print(f"Columns: {mv_df.columns.tolist()}")

### Gemma 4

In [ ]:
# ─────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────
OLLAMA_MODEL   = "gemma4:e4b"

# TRIGGER FAILSAFE HERE: Stop execution immediately if Ollama is down
check_ollama_ready(OLLAMA_MODEL)

OUTPUT_CSV     = rf"d:\Users\Desktop\Thesis\MathTrans\mathvision_gemma4translation.csv" 

# ─────────────────────────────────────────────────
# RESUME SUPPORT — skip already translated rows
# ─────────────────────────────────────────────────
columns = [
    "id", "question", "arabic_question" ,"translation_time_sec"
]

if Path(OUTPUT_CSV).exists():
    # Force 'id' to be read as a string to prevent type mismatches
    existing = pd.read_csv(OUTPUT_CSV, dtype={"id": str})
    
    # Create the set of done IDs
    done_ids = set(existing["id"].dropna().tolist())
    print(f"Resuming: {len(done_ids):,} rows already translated, skipping them.")
else:
    existing = None
    done_ids = set()
    pd.DataFrame(columns=columns).to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print("Starting fresh — output CSV created.")
    
# ─────────────────────────────────────────────────
# MAIN LOOP
# ─────────────────────────────────────────────────
SAMPLE_SIZE = None   # ← Set to an integer to process only the first N rows, or None to process ALL rows

# Force the main dataframe's ID column to string before comparing
rows_to_process = mv_df[~mv_df["id"].astype(str).isin(done_ids)].copy()

# (Optional) Re-add your sample size logic if you still want to use it
if SAMPLE_SIZE is not None:
    rows_to_process = rows_to_process.head(SAMPLE_SIZE)
    print(f"Sample mode: processing first {SAMPLE_SIZE} rows.")

total = len(rows_to_process)
print(f"Translating {total:,} rows...\n")

BATCH_SIZE = 5   # Append to CSV every N rows to avoid data loss on interruption
batch = []

for i, (idx, row) in enumerate(rows_to_process.iterrows(), start=1):
    # Extract columns from MathVision parquet
    row_id      = row["id"]
    question    = str(row["question"])
    
    # Translate only the question column
    arabic_question, elapsed = translate_to_arabic(question)

    batch.append({
        "id": row_id,
        "question": question,
        "arabic_question": arabic_question,
        "translation_time_sec": round(elapsed, 2)
    })

    # Progress print
    print(f"[{i}/{total}] id={row_id} | {elapsed:.2f}s | Q: {question[:50]}...")

    # Flush batch to disk
    if i % BATCH_SIZE == 0 or i == total:
        batch_df = pd.DataFrame(batch)[columns] 
        batch_df.to_csv(OUTPUT_CSV, mode="a", header=False, index=False, encoding="utf-8-sig")
        batch.clear()
        print(f"  → Saved {i:,}/{total:,} rows to {OUTPUT_CSV}")

print("\n✅ Translation complete!")


### aya

In [ ]:
# ─────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────
OLLAMA_MODEL   = "aya-expanse"

# TRIGGER FAILSAFE HERE: Stop execution immediately if Ollama is down
check_ollama_ready(OLLAMA_MODEL)

OUTPUT_CSV     = rf"d:\Users\Desktop\Thesis\MathTrans\mathvision_aya-expansetranslation.csv" 

# ─────────────────────────────────────────────────
# RESUME SUPPORT — skip already translated rows
# ─────────────────────────────────────────────────
columns = [
    "id", "question", "arabic_question" ,"translation_time_sec"
]

if Path(OUTPUT_CSV).exists():
    # Force 'id' to be read as a string to prevent type mismatches
    existing = pd.read_csv(OUTPUT_CSV, dtype={"id": str})
    
    # Create the set of done IDs
    done_ids = set(existing["id"].dropna().tolist())
    print(f"Resuming: {len(done_ids):,} rows already translated, skipping them.")
else:
    existing = None
    done_ids = set()
    pd.DataFrame(columns=columns).to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print("Starting fresh — output CSV created.")
    
# ─────────────────────────────────────────────────
# MAIN LOOP
# ─────────────────────────────────────────────────
SAMPLE_SIZE = None   # ← Set to an integer to process only the first N rows, or None to process ALL rows

# Force the main dataframe's ID column to string before comparing
rows_to_process = mv_df[~mv_df["id"].astype(str).isin(done_ids)].copy()

# (Optional) Re-add your sample size logic if you still want to use it
if SAMPLE_SIZE is not None:
    rows_to_process = rows_to_process.head(SAMPLE_SIZE)
    print(f"Sample mode: processing first {SAMPLE_SIZE} rows.")

total = len(rows_to_process)
print(f"Translating {total:,} rows...\n")

BATCH_SIZE = 5   # Append to CSV every N rows to avoid data loss on interruption
batch = []

for i, (idx, row) in enumerate(rows_to_process.iterrows(), start=1):
    # Extract columns from MathVision parquet
    row_id      = row["id"]
    question    = str(row["question"])
    
    # Translate only the question column
    arabic_question, elapsed = translate_to_arabic(question)

    batch.append({
        "id": row_id,
        "question": question,
        "arabic_question": arabic_question,
        "translation_time_sec": round(elapsed, 2)
    })

    # Progress print
    print(f"[{i}/{total}] id={row_id} | {elapsed:.2f}s | Q: {question[:50]}...")

    # Flush batch to disk
    if i % BATCH_SIZE == 0 or i == total:
        batch_df = pd.DataFrame(batch)[columns] 
        batch_df.to_csv(OUTPUT_CSV, mode="a", header=False, index=False, encoding="utf-8-sig")
        batch.clear()
        print(f"  → Saved {i:,}/{total:,} rows to {OUTPUT_CSV}")

print("\n✅ Translation complete!")


### qwen3

In [ ]:
# ─────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────
OLLAMA_MODEL   = "qwen3:8b"

# TRIGGER FAILSAFE HERE: Stop execution immediately if Ollama is down
check_ollama_ready(OLLAMA_MODEL)

OUTPUT_CSV     = rf"d:\Users\Desktop\Thesis\MathTrans\mathvision_qwen3translation.csv" 

# ─────────────────────────────────────────────────
# RESUME SUPPORT — skip already translated rows
# ─────────────────────────────────────────────────
columns = [
    "id", "question", "arabic_question" ,"translation_time_sec"
]

if Path(OUTPUT_CSV).exists():
    # Force 'id' to be read as a string to prevent type mismatches
    existing = pd.read_csv(OUTPUT_CSV, dtype={"id": str})
    
    # Create the set of done IDs
    done_ids = set(existing["id"].dropna().tolist())
    print(f"Resuming: {len(done_ids):,} rows already translated, skipping them.")
else:
    existing = None
    done_ids = set()
    pd.DataFrame(columns=columns).to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print("Starting fresh — output CSV created.")
    
# ─────────────────────────────────────────────────
# MAIN LOOP
# ─────────────────────────────────────────────────
SAMPLE_SIZE = None   # ← Set to an integer to process only the first N rows, or None to process ALL rows

# Force the main dataframe's ID column to string before comparing
rows_to_process = mv_df[~mv_df["id"].astype(str).isin(done_ids)].copy()

# (Optional) Re-add your sample size logic if you still want to use it
if SAMPLE_SIZE is not None:
    rows_to_process = rows_to_process.head(SAMPLE_SIZE)
    print(f"Sample mode: processing first {SAMPLE_SIZE} rows.")

total = len(rows_to_process)
print(f"Translating {total:,} rows...\n")

BATCH_SIZE = 5   # Append to CSV every N rows to avoid data loss on interruption
batch = []

for i, (idx, row) in enumerate(rows_to_process.iterrows(), start=1):
    # Extract columns from MathVision parquet
    row_id      = row["id"]
    question    = str(row["question"])
    
    # Translate only the question column
    arabic_question, elapsed = translate_to_arabic(question)

    batch.append({
        "id": row_id,
        "question": question,
        "arabic_question": arabic_question,
        "translation_time_sec": round(elapsed, 2)
    })

    # Progress print
    print(f"[{i}/{total}] id={row_id} | {elapsed:.2f}s | Q: {question[:50]}...")

    # Flush batch to disk
    if i % BATCH_SIZE == 0 or i == total:
        batch_df = pd.DataFrame(batch)[columns] 
        batch_df.to_csv(OUTPUT_CSV, mode="a", header=False, index=False, encoding="utf-8-sig")
        batch.clear()
        print(f"  → Saved {i:,}/{total:,} rows to {OUTPUT_CSV}")

print("\n✅ Translation complete!")

### Finale

In [ ]:
import pandas as pd

# Define your file path
# CSV_PATH = r"d:\Users\Desktop\Thesis\MathTrans\mathvision_gemma4translation.csv"
# CSV_PATH = r"d:\Users\Desktop\Thesis\MathTrans\mathvision_aya-expansetranslation.csv"
CSV_PATH = r"d:\Users\Desktop\Thesis\MathTrans\mathvision_qwen3translation.csv"

# Load the CSV file
df = pd.read_csv(CSV_PATH)

# Condition 1: The original English 'question' DOES end with a standard '?'
is_english_question = df["question"].fillna("").astype(str).str.strip().str.endswith("?")

# Condition 2: The translated 'arabic_question' DOES NOT end with an Arabic '؟'
missing_arabic_qm = ~df["arabic_question"].fillna("").astype(str).str.strip().str.endswith("؟")

# Combine both conditions using the '&' (AND) operator
# This filters for rows where BOTH conditions are true at the same time
problematic_rows = df[is_english_question & missing_arabic_qm]

# Get the total count and the specific IDs
missing_count = len(problematic_rows)
problematic_ids = problematic_rows["id"].tolist()

# Print the results
print(f"Total rows where English had '?' but Arabic is missing '؟': {missing_count}")

if missing_count > 0:
    print("\nList of IDs with missing question marks:")
    # Print the IDs comma-separated for easy reading
    print(", ".join(map(str, problematic_ids)))

# Define the exact name of your time column
TIME_COL = "translation_time_sec" 

# 1. Isolate the time column and drop any completely blank/missing rows
times = df[TIME_COL].dropna()

# 2. Calculate the Interquartile Range (IQR)
Q1 = times.quantile(0.25) # 25th percentile
Q3 = times.quantile(0.75) # 75th percentile
IQR = Q3 - Q1

# 3. Define the mathematical bounds for what constitutes an "outlier"
# The 1.5 multiplier is the industry standard for identifying outliers
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# 4. Filter the data to ONLY keep values inside the bounds
filtered_times = times[(times >= lower_bound) & (times <= upper_bound)]

# 5. Calculate the true average without the outliers
true_average = filtered_times.mean()

# Print the results clearly
outliers_removed = len(times) - len(filtered_times)

print("--- Translation Time Analysis ---")
print(f"Original row count  : {len(times):,}")
print(f"Rows kept (clean)   : {len(filtered_times):,}")
print(f"Outliers removed    : {outliers_removed:,}")
print(f"---------------------------------")
print(f"Raw Average (w/ outliers)   : {times.mean():.2f} seconds")
print(f"Clean Average (no outliers) : {true_average:.2f} seconds")

In [ ]:
# ─────────────────────────────────────────────────
# FINALE — Merge translations into a new parquet file
# ─────────────────────────────────────────────────
import pandas as pd

PARQUET_IN  = r"D:\Users\Desktop\Thesis\drive datsets\mathvision.parquet"
CSV_IN      = r"d:\Users\Desktop\Thesis\MathTrans\mathvision_gemma4translation.csv"
PARQUET_OUT = r"d:\Users\Desktop\Thesis\mathvision_arabic.parquet"

# Load both files
df_pq  = pd.read_parquet(PARQUET_IN)
df_csv = pd.read_csv(CSV_IN, dtype=str)

# Keep only the translation column we need from the CSV
df_trans = df_csv[["id", "question", "arabic_question"]].copy()

# 1. Clean strings using TEMPORARY columns to ensure perfect matches
# This protects your original data from being altered in any way.
df_pq["_merge_id"] = df_pq["id"].astype(str).str.strip()
df_pq["_merge_question"] = df_pq["question"].astype(str).str.strip()

df_trans["_merge_id"] = df_trans["id"].astype(str).str.strip()
df_trans["_merge_question"] = df_trans["question"].astype(str).str.strip()

# 2. Add a temporary counter to distinguish duplicates
df_pq["occurrence"] = df_pq.groupby(["_merge_id", "_merge_question"]).cumcount()
df_trans["occurrence"] = df_trans.groupby(["_merge_id", "_merge_question"]).cumcount()

# 3. Merge using the temporary keys
# We drop the CSV's original id/question so they don't overwrite the Parquet's keys
df_merged = df_pq.merge(
    df_trans.drop(columns=["id", "question"]), 
    on=["_merge_id", "_merge_question", "occurrence"], 
    how="left"
)

# 4. Clean up all temporary columns
df_merged = df_merged.drop(columns=["_merge_id", "_merge_question", "occurrence"])

print(f"Original Parquet rows : {len(df_pq):,}")
print(f"Merged Parquet rows   : {len(df_merged):,} (Should match original!)")

matched = df_merged["arabic_question"].notna().sum()
print(f"Matched translations  : {matched:,} / {len(df_merged):,}")

# Save as parquet
df_merged.to_parquet(PARQUET_OUT, index=False, compression="zstd")
print(f"\n✅ Saved to: {PARQUET_OUT}")

# ─────────────────────────────────────────────────
# VERIFICATION STEP — Ensure original data is untouched
# ─────────────────────────────────────────────────
print("\n🔍 Running strict data verification...")

# Load both files straight from the hard drive for an unbiased check
df_original_check = pd.read_parquet(PARQUET_IN)
df_new_check = pd.read_parquet(PARQUET_OUT)

# Get the list of original columns
original_cols = df_original_check.columns.tolist()

try:
    # assert_frame_equal strictly checks that values, datatypes, and order are 100% identical
    pd.testing.assert_frame_equal(
        df_original_check[original_cols], 
        df_new_check[original_cols],
        check_dtype=True,
        check_exact=True
    )
    print("✅ SUCCESS: All original columns are 100% identical and untouched!")
except AssertionError as e:
    print("❌ WARNING: The original data was altered during the process.")
    print(f"Details: {e}")

df = pd.read_parquet(PARQUET_OUT)
print(f"Loaded {len(df):,} rows")
print(f"Columns: {df.columns.tolist()}")